In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, ClassifierMixin

# Data Preparation & Split

In [16]:
import pandas as pd

iris = load_iris()
X, y = iris.data, iris.target

# Scaling is highly recommended for gradient descent and the 'saga' solver
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples\n")

# Visualize the scaled dataset
df = pd.DataFrame(X_scaled, columns=iris.feature_names)
# Map target integers to target names for better readability
target_names_map = {i: name for i, name in enumerate(iris.target_names)}
df['target'] = pd.Series(y).map(target_names_map)

print("Scaled Iris Dataset (first 5 rows):")
print(df.head())

Training set: 112 samples
Test set: 38 samples

Scaled Iris Dataset (first 5 rows):
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0          -0.900681          1.019004          -1.340227         -1.315444   
1          -1.143017         -0.131979          -1.340227         -1.315444   
2          -1.385353          0.328414          -1.397064         -1.315444   
3          -1.506521          0.098217          -1.283389         -1.315444   
4          -1.021849          1.249201          -1.340227         -1.315444   

   target  
0  setosa  
1  setosa  
2  setosa  
3  setosa  
4  setosa  


# (a) Logistic Regression From Scratch

In [17]:
class CustomLogisticRegression(BaseEstimator, ClassifierMixin):
    def __init__(self, learning_rate=0.1, epochs=2000):
        self.learning_rate = learning_rate
        self.epochs = epochs

    def fit(self, X, y):
        m, n = X.shape
        self.classes_ = np.unique(y)
        k = len(self.classes_)

        # Initialize weights and bias
        self.W = np.zeros((n, k))
        self.b = np.zeros(k)

        # One-hot encode the target labels
        Y_oh = np.zeros((m, k))
        for i, c in enumerate(y):
            Y_oh[i, np.where(self.classes_ == c)] = 1

        # Gradient Descent
        for _ in range(self.epochs):
            # Compute scores: Z = XW + b
            Z = np.dot(X, self.W) + self.b

            # Softmax probabilities
            Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
            exp_Z = np.exp(Z_shifted)
            probabilities = exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

            # Compute gradients
            dW = (1 / m) * np.dot(X.T, (probabilities - Y_oh))
            db = (1 / m) * np.sum(probabilities - Y_oh, axis=0)

            # Update parameters
            self.W -= self.learning_rate * dW
            self.b -= self.learning_rate * db

        return self

    def predict(self, X):
        Z = np.dot(X, self.W) + self.b
        return self.classes_[np.argmax(Z, axis=1)]

# (b) Logistic Regression using scikit-learn

In [18]:
sklearn_lr = LogisticRegression(solver='saga', penalty=None, max_iter=2000, random_state=42)

# (c) Cross-Validation (3-folds)

In [19]:
scoring_metrics = ['accuracy', 'precision_macro', 'recall_macro']

# Initialize our custom model
custom_lr = CustomLogisticRegression(learning_rate=0.5, epochs=2000)
cv_results_custom = cross_validate(custom_lr, X_train, y_train, cv=3, scoring=scoring_metrics)

print("Custom Implementation (From Scratch):")
print(f"Mean Accuracy:  {np.mean(cv_results_custom['test_accuracy']):.4f}")
print(f"Mean Precision: {np.mean(cv_results_custom['test_precision_macro']):.4f}")
print(f"Mean Recall:    {np.mean(cv_results_custom['test_recall_macro']):.4f}\n")

# Cross-validate Scikit-Learn Model
cv_results_sklearn = cross_validate(sklearn_lr, X_train, y_train, cv=3, scoring=scoring_metrics)
print("Scikit-Learn Implementation (saga solver):")
print(f"Mean Accuracy:  {np.mean(cv_results_sklearn['test_accuracy']):.4f}")
print(f"Mean Precision: {np.mean(cv_results_sklearn['test_precision_macro']):.4f}")
print(f"Mean Recall:    {np.mean(cv_results_sklearn['test_recall_macro']):.4f}")

Custom Implementation (From Scratch):
Mean Accuracy:  0.9552
Mean Precision: 0.9643
Mean Recall:    0.9529

Scikit-Learn Implementation (saga solver):
Mean Accuracy:  0.9464
Mean Precision: 0.9574
Mean Recall:    0.9487


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
